# Movie Dataset EDA


2 data files: **data.npy** (for movie records) and **images.npz** (for poster images)

In [3]:
# Load packages and libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
import hashlib

In [ ]:
from pathlib import Path
import os

try:
    from google.colab import drive

    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")

    default_dir = Path(
        "/content/drive/MyDrive/Multi Modal - Project 2.2"
    )

except ImportError:
    default_dir = Path.cwd() / "data"

PROJECT_DIR = Path(
    os.environ.get("MOVIE_PROJECT_DIR", str(default_dir))
).expanduser()

assert (PROJECT_DIR / "data.npy").is_file()
assert (PROJECT_DIR / "images.npz").is_file()

print("Dataset directory:", PROJECT_DIR)


In [4]:
# Genre list
genres = ["Drama", "Comedy", "Romance", "Thriller", "Crime", "Action", "Adventure", "Horror",
          "Documentary", "Mystery", "Sci-Fi", "Music", "Fantasy", "Family", "Biography", "War",
          "History", "Animation", "Musical", "Western", "Sport", "Short", "Film-Noir"]

#### 1. Load files and inspect structure

In [5]:
# Load movie records and images
records = np.load(PROJECT_DIR / "data.npy", allow_pickle=True)
images = np.load(PROJECT_DIR / "images.npz")["images"]

# Record and image shape
print("Records:", records.shape)
print("Images:", images.shape)

FileNotFoundError: [Errno 2] No such file or directory: 'D:\\Downloads\\data.npy'

In [ ]:
# First movie record
print(records[0])
print(images[0])

In [ ]:
# Movie record + poster
for i in [0, 1, 2]:
    row_id, label_vector, summary = records[i]
    genre_names = [genres[j] for j, value in enumerate(label_vector) if value == 1]
    # Poster images
    plt.figure(figsize=(3, 4))
    plt.imshow(np.moveaxis(images[i], 0, -1))
    plt.axis("off")
    plt.show()
    # Movie records
    print(f"Row ID: {row_id}")
    print(f"Genres: {', '.join(genre_names)}")
    print(f"Plot Summary: {summary[:500]}{'...' if len(summary) > 500 else ''}")
    print()

* There are 25,959 records and 25,959 images. Each record contains a row ID, a 23-value genre vector, and plot summary. Posters are of (3,256,160) array shape per image.

#### 2. Check missing values and poster alignment

In [ ]:
ids = records[:,0]
labels = np.stack([np.asarray(row, dtype=int) for row in records[:,1]])
summaries = [str(row) for row in records[:,2]]

# Check record and image count
print("IDs sequential: ", ids.tolist() == list(range(len(records))))
print("Record and image counts match: ", len(records) == len(images))

# Check if all records have plot summaries, valid labels
print("Empty plots: ", sum(not summary.strip() for summary in summaries))
print("Labels are binary: ", np.isin(labels, [0,1]).all())
print("Movies with no labels: ", (labels.sum(axis=1) == 0).sum())

* IDs run sequentially from 0-25958, the image and record counts match, plot summaries are non-empty, and labels are binary. The intended match is by row position: records[i] goes with images[i].

#### 3. Examine genre frequencies

In [ ]:
# Genre frequencies
genre_summary = pd.DataFrame({
    "genre": genres,
    "count": labels.sum(axis=0),
    "proportion": labels.mean(axis=0)*100
}).sort_values("count", ascending=False)

display(genre_summary.style.format({"proportion": "{:.1f}%"}))

# Plot genre distribution
genre_plot = genre_summary.sort_values("count")

plt.figure(figsize=(9, 7))
bars = plt.barh(genre_plot["genre"], genre_plot["count"])
for bar, proportion in zip(bars, genre_plot["proportion"]):
    plt.text(
        bar.get_width() + 30,
        bar.get_y() + bar.get_height() / 2,
        f"{proportion:.1f}%",
        va="center"
    )
plt.title("Movie Genre Distribution")
plt.xlim(0, genre_plot["count"].max() * 1.15)
ax = plt.gca()
ax.tick_params(axis="x", which="both", length=0, labelbottom=False)
ax.tick_params(axis="y", which="both", length=0, labelleft=True)
for spine in ax.spines.values():
    spine.set_visible(False)
plt.tight_layout()
plt.show()


* Drama is the most frequent label (53.8%), while Film-Noir is the least frequent (1.3%).

#### 4. Check summary lengths and duplicates

In [ ]:
# Check summary length
summary_len = np.array([len(summary.split()) for summary in summaries])
print(f"Min: {np.min(summary_len)}")
print(f"Mean: {np.round(np.mean(summary_len),0)}")
print(f"Median: {np.median(summary_len)}")
print(f"Max: {np.max(summary_len)}")

In [ ]:
# Check summary duplicates
counts = Counter(summaries)
duplicate_summaries = [(text, count) for text, count in counts.items() if count > 1]
print(f"Number of repeated summaries: {len(duplicate_summaries):,}\n")
for text, count in duplicate_summaries:
    print(f"Repeat {count} times: {text}")

* Plot summary length has: min = 1, max = 1887, mean = 124, median = 107.
* There are 11 repeated plot summaries found.

#### 5. Check poster quality and duplicates

In [ ]:
# Check poster quality
print("Shape:", images.shape)
print("Data type:", images.dtype)
print("Pixel range:", images.min(), "to", images.max())
print("All-zero images:", np.sum(images.reshape(len(images), -1).max(axis=1) == 0))

In [ ]:
# Check poster duplicates

# Group row indices by identical iamge content
groups = defaultdict(list)
for i, image in enumerate(images):
    groups[hashlib.md5(image.tobytes()).digest()].append(i)
duplicates = [rows for rows in groups.values() if len(rows) > 1]
print(f"Duplicate poster image groups: {len(duplicates)}")
print(f"Duplicate row ids: {duplicates}")

# Show the first duplicate pair
if duplicates:
    rows = duplicates[0][:2]
    fig, axes = plt.subplots(1, 2, figsize=(6, 4))
    for ax, i in zip(axes, rows):
        ax.imshow(np.moveaxis(images[i], 0, -1))
        ax.set_title(f"Row {i}")
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No exact duplicate posters found.")

* Poster images are of shape (25959, 3, 256, 160), with pixel values ranging from 0 to 255, and there is no all-zero images.
* There are 63 duplicate poster image groups found.

#### 6. Preprocessing and Data Splitting
##### 6.1. Grouping Duplicate Records

In [ ]:
with np.load(PROJECT_DIR / "images.npz") as f:
    print("Arrays in images.npz:", f.files)

print("Columns in records:", records.shape[1])

In [ ]:
from collections import defaultdict, Counter

plot_groups = defaultdict(list)

for i, text in enumerate(summaries):
    plot_groups[text].append(i)

duplicate_plot_groups = [
    rows for rows in plot_groups.values()
    if len(rows) > 1
]

print("Number of groups with identical plots:", len(duplicate_plot_groups))

In [ ]:
from collections import Counter

parent = np.arange(len(records))

def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i

for rows in duplicate_plot_groups + duplicates:
    for i in rows[1:]:
        parent[find(i)] = find(rows[0])

group_ids = np.array([find(i) for i in range(len(records))])

group_sizes = Counter(group_ids)

print("Total number of groups:", len(group_sizes))
print("Number of groups with multiple records:",
      sum(size > 1 for size in group_sizes.values()))

##### 6.2. Creating Fixed Train/Validation/Test Splits

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
import numpy as np

seed = 42
all_indices = np.arange(len(records))

#Split into train + validation (70%) and test (30%)
split_1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state= seed)
train_val_pos, test_pos = next(split_1.split(all_indices, groups=group_ids))
train_val_idx = all_indices[train_val_pos]
test_idx = all_indices[test_pos]

#Split train + validation into train (60%) and validation (10%)
split_2 = GroupShuffleSplit(n_splits=1, test_size=1/7, random_state=seed)
train_pos, val_pos = next(split_2.split(train_val_idx, groups=group_ids[train_val_idx]))
train_idx = train_val_idx[train_pos]
val_idx = train_val_idx[val_pos]

# Display split sizes and proportions
print("Training records:", len(train_idx))
print("Validation records:", len(val_idx))
print("Testing records:", len(test_idx))

print("\nSplit proportions:")
print(f"Training: {len(train_idx)/len(records):.2%}")
print(f"Validation: {len(val_idx)/len(records):.2%}")
print(f"Testing: {len(test_idx)/len(records):.2%}")

print("\nTotal records:", len(train_idx) + len(val_idx) + len(test_idx))

##### 6.3. Checking Data Leakage

In [ ]:

# Identify the duplicate groups in each split
train_groups = set(group_ids[train_idx])
val_groups = set(group_ids[val_idx])
test_groups = set(group_ids[test_idx])

# Check for overlapping groups
train_val_overlap = train_groups.intersection(val_groups)
train_test_overlap = train_groups.intersection(test_groups)
val_test_overlap = val_groups.intersection(test_groups)

print("Train-Validation overlap:", len(train_val_overlap))
print("Train-Test overlap:", len(train_test_overlap))
print("Validation-Test overlap:", len(val_test_overlap))

# Verify all records are included exactly once
all_split_indices = np.concatenate([
    train_idx, val_idx, test_idx
])

print("\nTotal records assigned:", len(all_split_indices))
print("Unique records assigned:", len(np.unique(all_split_indices)))

# Final verification
assert len(train_val_overlap) == 0
assert len(train_test_overlap) == 0
assert len(val_test_overlap) == 0
assert len(np.unique(all_split_indices)) == len(records)

##### 6.4. Verifying Split Distributions

In [ ]:
# Extract the 23 binary genre labels
labels = np.stack(records[:, 1]).astype(int)

# Calculate genre prevalence in each split
train_dist = labels[train_idx].mean(axis=0)
val_dist = labels[val_idx].mean(axis=0)
test_dist = labels[test_idx].mean(axis=0)
overall_dist = labels.mean(axis=0)

# Use genre names from the original EDA if available
if "genre_names" in globals() and len(genre_names) == labels.shape[1]:
    names = list(genre_names)
else:
    names = [f"Genre_{i}" for i in range(labels.shape[1])]

# Create a distribution comparison table
distribution_df = pd.DataFrame({
    "Genre": names,
    "Overall (%)": overall_dist * 100,
    "Train (%)": train_dist * 100,
    "Validation (%)": val_dist * 100,
    "Test (%)": test_dist * 100
})

# Compare each split with the overall distribution
split_distributions = np.column_stack([
    train_dist,
    val_dist,
    test_dist
])

# Calculate the maximum deviation in percentage points
deviations = np.abs(
    split_distributions - overall_dist[:, None]
)

distribution_df["Max deviation (pp)"] = (
    np.max(deviations, axis=1) * 100
)

# Round results to 2 decimal places
distribution_df = distribution_df.round(2)

# Display the results
print("Genre Distribution Across Dataset Splits:\n")
print(distribution_df.to_string(index=False))

print("\nMaximum genre prevalence deviation (percentage points):",
      distribution_df["Max deviation (pp)"].max())

# Check whether every genre is represented in each split
print("\nNumber of genres present in each split:")
print("Training:", np.sum(labels[train_idx].sum(axis=0) > 0))
print("Validation:", np.sum(labels[val_idx].sum(axis=0) > 0))
print("Testing:", np.sum(labels[test_idx].sum(axis=0) > 0))

In [ ]:
# Save the fixed train/validation/test indices
np.savez_compressed(
    "split_indices.npz",
    train_idx=train_idx,
    val_idx=val_idx,
    test_idx=test_idx
)

# Reload and verify the saved file
with np.load("split_indices.npz") as split_data:
    assert np.array_equal(split_data["train_idx"], train_idx)
    assert np.array_equal(split_data["val_idx"], val_idx)
    assert np.array_equal(split_data["test_idx"], test_idx)

print("Training records:", len(train_idx))
print("Validation records:", len(val_idx))
print("Testing records:", len(test_idx))


##### 6.5. Text Preprocessing for DistilBERT

In [ ]:
# Install transformers if needed
%pip install -q "transformers>=4.45,<5"

from transformers import AutoTokenizer

# Load the DistilBERT tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    "distilbert/distilbert-base-uncased"
)

# Extract plot summaries
summaries = [str(text) for text in records[:, 2]]

# Tokenize without truncation or padding
token_lengths = []
for i in range(0, len(summaries), 256):
    batch = summaries[i:i + 256]

    encoded = tokenizer(
        batch,
        truncation=False,
        padding=False,
        add_special_tokens=True
    )
    token_lengths.extend(len(ids) for ids in encoded["input_ids"])
token_lengths = np.array(token_lengths)

# Calculate descriptive statistics
print("Total plots:", len(token_lengths))
print("Median token length:", np.median(token_lengths))
print("Mean token length:", round(np.mean(token_lengths), 2))
print("Maximum token length:", np.max(token_lengths))

# Compare truncation at 256 and 512 tokens
for max_len in [256, 512]:
    truncated = np.sum(token_lengths > max_len)
    percentage = truncated / len(token_lengths) * 100

    print(f"\nMaximum sequence length: {max_len}")
    print("Plots requiring truncation:", truncated)
    print(f"Percentage requiring truncation: {percentage:.2f}%")

- A maximum sequence length of 512 tokens preserves the full content of more plot summaries than 256 tokens, reducing truncation from 10.28% to 0.47%. However, longer sequences may increase computational time and GPU memory usage. Therefore, 256 tokens will be used for the initial GPU feasibility test due to its lower computational cost. If time permits, both settings will be compared in terms of runtime and memory consumption before selecting the final configuration. The impact on model performance (Macro F1) remains to be evaluated.

**Tokenization, Padding and Truncation Test**

In [ ]:
# Select 8 plots from the training set
sample_texts = [summaries[i] for i in train_idx[:8]]

# Apply DistilBERT tokenization
encoded = tokenizer(
    sample_texts,
    padding="max_length",
    truncation=True,
    max_length=256,
    return_tensors="pt"
)

# Inspect tokenized inputs
print("Input IDs shape:", encoded["input_ids"].shape)
print("Attention mask shape:", encoded["attention_mask"].shape)

# Verify input dimensions
assert encoded["input_ids"].shape == (8, 256)
assert encoded["attention_mask"].shape == (8, 256)

# Check actual token lengths after preprocessing
print("\nNon-padding tokens per plot:")
print(encoded["attention_mask"].sum(dim=1).tolist())

##### 6.6. Image Preprocessing for CLIP

In [ ]:
#Testing CLIP Image Preprocessing
from transformers import AutoImageProcessor

# Load CLIP image processor
image_processor = AutoImageProcessor.from_pretrained("openai/clip-vit-base-patch32")

# Select 8 posters from the training set
sample_indices = train_idx[:8]

# Convert images from CHW to HWC format
sample_images = [np.transpose(images[i], (1, 2, 0)) for i in sample_indices]

# Apply CLIP preprocessing
processed_images = image_processor(images=sample_images, return_tensors="pt")

# Inspect original and processed image dimensions
print("Original image shape:", images[sample_indices[0]].shape)
print("Converted image shape:", sample_images[0].shape)
print("Processed pixel values shape:", processed_images["pixel_values"].shape)

# Verify expected dimensions
assert processed_images["pixel_values"].shape == (8, 3, 224, 224)

# Check numerical validity
assert np.isfinite(processed_images["pixel_values"].numpy()).all()

- CLIP image preprocessing successfully converted poster images from the original channel-first format (3, 256, 160) to model-ready tensors (3, 224, 224). All 8 sampled posters were processed without invalid numerical values, confirming compatibility with the CLIP ViT-B/32 input requirements. This supports the feasibility of extracting visual features for subsequent multimodal classification.

#### 7. GPU Feasibility Testing
##### 7.1. GPU Environment Setup

In [1]:

import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    device = torch.device("cuda")
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "Total GPU memory (GiB):",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2)
    )
else:
    device = torch.device("cpu")
    print("GPU is not available.")


PyTorch version: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
Total GPU memory (GiB): 14.56


In [2]:

%pip install -q "transformers>=4.45,<5"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 75.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.29.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [1]:

import torch
import transformers
import huggingface_hub

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Hugging Face Hub:", huggingface_hub.__version__)
print("CUDA available:", torch.cuda.is_available())


PyTorch: 2.11.0+cu130
Transformers: 4.57.6
Hugging Face Hub: 0.36.2
CUDA available: True


In [2]:

from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoImageProcessor,
    CLIPVisionModel
)

print("All required Transformers classes imported successfully.")


All required Transformers classes imported successfully.


##### 7.2. Loading Pretrained Models

In [3]:

# Select GPU
device = torch.device("cuda")

# Load pretrained DistilBERT
text_tokenizer = AutoTokenizer.from_pretrained(
    "distilbert/distilbert-base-uncased"
)

text_model = AutoModel.from_pretrained(
    "distilbert/distilbert-base-uncased"
).to(device)

text_model.eval()

# Load pretrained CLIP vision encoder
image_processor = AutoImageProcessor.from_pretrained(
    "openai/clip-vit-base-patch32",
    use_fast=False
)

image_model = CLIPVisionModel.from_pretrained(
    "openai/clip-vit-base-patch32"
).to(device)

image_model.eval()

# Verify model devices
print("DistilBERT device:", next(text_model.parameters()).device)
print("CLIP Vision device:", next(image_model.parameters()).device)

print("\nBoth pretrained encoders loaded successfully.")


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

DistilBERT device: cuda:0
CLIP Vision device: cuda:0

Both pretrained encoders loaded successfully.
